# Step 3: Feature Engineering (Baseline)

WoE encoding, IV feature selection for structured features.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/src')

import pandas as pd
import matplotlib.pyplot as plt
from spark_utils import get_spark_session, save_parquet
from feature_engineering import calculate_woe, calculate_iv, select_features_by_iv, apply_woe_encoding

In [ ]:
SAMPLE_MODE = True

## 3.1 Load Cleaned Data

In [ ]:
spark = get_spark_session()
df = spark.read.parquet('/home/jovyan/work/data/parquet/accepted_cleaned.parquet')

# Convert to pandas for WoE calculation
df_pd = df.toPandas()

## 3.2 Define Candidate Features

In [ ]:
# Structured features for baseline model
candidate_features = [
    'loan_amnt', 'funded_amnt', 'int_rate', 'installment',
    'annual_inc', 'dti', 'delinq_2yrs', 'open_acc',
    'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'mort_acc', 'pub_rec_bankruptcies',
]

## 3.3 IV Feature Selection

In [ ]:
iv_results = select_features_by_iv(df_pd, candidate_features, threshold=0.02)
iv_results

In [ ]:
# IV ranking bar chart
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(iv_results['feature'], iv_results['iv'], color='steelblue')
ax.set_xlabel('Information Value')
ax.set_title('Feature Ranking by IV')
ax.axvline(x=0.1, color='orange', linestyle='--', label='Medium threshold')
ax.axvline(x=0.3, color='red', linestyle='--', label='Strong threshold')
ax.legend()
plt.tight_layout()
plt.savefig('/home/jovyan/work/outputs/figures/iv_ranking.png', dpi=150)
plt.show()

## 3.4 WoE Encoding

In [ ]:
selected_features = iv_results['feature'].tolist()

for feat in selected_features:
    df_pd = apply_woe_encoding(df_pd, feat)

woe_columns = [f'{feat}_woe' for feat in selected_features]
print(f'WoE encoded features: {woe_columns}')

## 3.5 Save Baseline Feature Matrix

In [ ]:
baseline_df = spark.createDataFrame(df_pd)
save_parquet(baseline_df, '/home/jovyan/work/data/parquet/features_baseline.parquet')
print('Baseline features saved.')